# Building Custom States and Operators

The [State](../apidoc/_autosummary/pulser.backend.State.rst) and [Operator](../apidoc/_autosummary/pulser.backend.Operator.rst) classes let you build arbitrary state and operator objects from scratch using the `from_state_amplitudes()` and `from_operator_repr()` methods. A common use case for this is many-body physics simulations, where you often want to:

- define 1- and 2-point correlation functions of Pauli operators (e.g. $\langle Z_i X_j \rangle_c$) between arbitrary sites,
- prepare reference initial states that are not easy to reach on the device. If a candidate state $|\psi\rangle$ turns out to be uninteresting for the your observable, there is no reason to spend time looking for a pulse sequence that prepares it.
- scale the number of qubits $L$ to look for trends that persist (or not) towards the thermodynamic limit.

This tutorial shows how to wrap the low-level `from_operator_repr()` and `from_state_amplitudes()` calls into small, reusable helpers that are parametrized by the system size, and how to use them to scan 2-point connected correlation functions across several candidate states and system sizes.

## Recap: representing operators and states

An operator is defined through a `FullOp`: a weighted sum of `TensorOp`,
each of which is a list of single-qudit `QuditOp` applied to disjoint sets of
qudits. A `QuditOp` is itself a mapping between `"ij"` strings (for
$|i\rangle\langle j|$, with `i`, `j` taken from `eigenstates`) and their
coefficient.

For a two-level system with `eigenstates = ("0", "1")`, the Pauli matrices are
written as:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pulser_simulation import QutipConfig

eigenstates = ("0", "1")

sigma_repr = {
    "x": {"01": 1.0, "10": 1.0},
    "y": {"01": -1.0j, "10": 1.0j},
    "z": {"00": 1.0, "11": -1.0},
}

`QutipConfig` is the `EmulationConfig` of the QuTiP-based emulator, and its
`state_type` and `operator_type` attributes give the concrete `State` and
`Operator` classes to instantiate. Any other backend config exposes the same
two attributes, so everything below is backend-agnostic.

## A size-parametrized helper class

Instead of repeating `eigenstates` and `n_qudits` at every call, it is
convenient to wrap them in a small class indexed by the number of qubits `L`.
The `corr_pauli` method below builds the operator for a product of identical
Pauli matrices acting on an arbitrary subset of sites (e.g.
`corr_pauli("x", 0, 2)` builds $X_0 X_2$), and `get_state` builds a few
reference states of size `L`.

In [ ]:
class SpinChain:
    """A qubit chain of a given size, with helpers to build Pauli-string
    operators and a few reference states."""

    def __init__(self, L, config_class=QutipConfig):
        self.L = L
        self.eigenstates = eigenstates
        self.state_class = config_class.state_type
        self.operator_class = config_class.operator_type

    def corr_pauli(self, a, *idxs):
        """Builds the operator for a Pauli string, e.g. corr_pauli("x", 0, 2)
        builds X_0 X_2. Does not support mixing directions, such as X_0 Z_1.
        """
        # use a set to remove duplicates, and to support a single list/tuple
        # of indices being passed instead of separate arguments
        if len(idxs) == 1 and not isinstance(idxs[0], int):
            idx = set(idxs[0])
        else:
            idx = set(idxs)

        return self.operator_class.from_operator_repr(
            eigenstates=self.eigenstates,
            n_qudits=self.L,
            operations=[(1.0, [(sigma_repr[a], idx)])],
        )

    def get_state(self, state_name):
        """Builds a reference state of size L."""
        if state_name == "down":
            # all qubits in |0>
            return self.state_class.from_state_amplitudes(
                eigenstates=self.eigenstates,
                amplitudes={"0" * self.L: 1.0},
            )

        elif state_name == "plus":
            # |+>^L : uniform superposition of all basis states
            dim = 2**self.L
            amplitudes = {
                "".join(
                    self.eigenstates[int(b)] for b in format(i, f"0{self.L}b")
                ): 1.0
                / np.sqrt(dim)
                for i in range(dim)
            }
            return self.state_class.from_state_amplitudes(
                eigenstates=self.eigenstates, amplitudes=amplitudes
            )

        elif state_name == "ghz":
            # (|00...0> + |11...1>) / sqrt(2)
            norm = 1.0 / np.sqrt(2)
            return self.state_class.from_state_amplitudes(
                eigenstates=self.eigenstates,
                amplitudes={"0" * self.L: norm, "1" * self.L: norm},
            )

        elif state_name == "dimer":
            # product of Bell pairs on sites (0,1), (2,3), ...
            if self.L % 2 != 0:
                raise ValueError("'dimer' state requires an even L.")
            n_pairs = self.L // 2
            norm = 1.0 / np.sqrt(2**n_pairs)
            amplitudes = {
                "".join(str((bits >> k) & 1) * 2 for k in range(n_pairs)): norm
                for bits in range(2**n_pairs)
            }
            return self.state_class.from_state_amplitudes(
                eigenstates=self.eigenstates, amplitudes=amplitudes
            )

        raise NotImplementedError(f"State {state_name} not implemented.")

`corr_pauli` and `get_state` both only depend on `self.L`, so the exact same
code can be reused to build operators and states for any system size, and
they are built directly with `from_operator_repr()` /
`from_state_amplitudes()`, so they remain compatible with remote backends.

In [ ]:
system = SpinChain(L=4)

x0 = system.corr_pauli("x", 0)  # X_0
z1z3 = system.corr_pauli("z", [1, 3])  # Z_1 Z_3, indices as a list also works

x0.to_qobj()

## Scanning connected correlation functions

A common quantity of interest in many-body physics is the connected 2-point
correlation function of an observable $O$,

$$C^{O}_{ij} = \langle O_i O_j \rangle - \langle O_i \rangle \langle O_j \rangle ,$$

which isolates genuine correlations between sites $i$ and $j$ from what is
expected from their individual averages ($C^O_{ii}$ is just the variance of
$O_i$). The following function builds the full $L\times L$ matrix of
connected correlators for a given Pauli direction and state, by combining
`corr_pauli()` (for the operators) with `Operator.expect()`:

In [ ]:
def connected_correlation_matrix(system, pauli, state):
    L = system.L
    one_point = np.array(
        [system.corr_pauli(pauli, i).expect(state).real for i in range(L)]
    )
    corr = np.zeros((L, L))
    for i in range(L):
        for j in range(L):
            two_point = (
                1.0  # <O_i^2> = 1 for Pauli operators
                if i == j
                else system.corr_pauli(pauli, i, j).expect(state).real
            )
            corr[i, j] = two_point - one_point[i] * one_point[j]
    return corr

This is enough to compare candidate initial states before worrying about how
(or whether) to prepare them on the device. Below, the $ZZ$ connected
correlations of three states are compared over a few system sizes $L$:

- `"plus"`, a product state ($|+\rangle^{\otimes L}$): no genuine
  correlations are expected between any two sites,
- `"ghz"`: a maximally entangled state, expected to show perfect long-range
  order between *any* pair of sites, regardless of $L$,
- `"dimer"`: a product of independent Bell pairs, expected to show
  correlations only *within* a pair, i.e. a finite correlation length that
  does not grow with $L$.

In [ ]:
system_sizes = [4, 6, 8]
state_names = ["plus", "ghz", "dimer"]

fig, axes = plt.subplots(
    len(state_names), len(system_sizes), figsize=(9, 7), squeeze=False
)
for row, name in enumerate(state_names):
    for col, L in enumerate(system_sizes):
        system = SpinChain(L)
        state = system.get_state(name)
        corr = connected_correlation_matrix(system, "z", state)
        ax = axes[row][col]
        im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
        ax.set_title(f"{name}, L={L}")
        ax.set_xlabel("site $j$")
        if col == 0:
            ax.set_ylabel("site $i$")
fig.colorbar(im, ax=axes, shrink=0.6, label=r"$C^{zz}_{ij}$")
plt.show()

The `"plus"` state shows no connected correlations at all, the `"dimer"`
state shows correlations that stay confined to each Bell pair irrespective of
$L$, and the `"ghz"` state shows correlations of magnitude 1 between *every*
pair of sites, no matter how far apart or how large the system is. Plotting
the correlation as a function of distance $|i-j|$ for a fixed, larger $L$
makes the contrast between a finite correlation length and long-range order
even clearer:

In [ ]:
L = 10
system = SpinChain(L)
distances = np.arange(1, L)

plt.figure()
for name in state_names:
    state = system.get_state(name)
    corr = connected_correlation_matrix(system, "z", state)
    decay = [corr[0, d] for d in distances]
    plt.plot(distances, decay, marker="o", label=name)
plt.xlabel(r"distance $|i-j|$")
plt.ylabel(r"$C^{zz}_{0,i}$")
plt.title(f"L={L}")
plt.legend()
plt.show()

Because `SpinChain` only depends on `L`, this whole comparison can be
repeated for larger and larger system sizes to check whether a trend (e.g. a
correlation length, or the persistence of long-range order) survives towards
the thermodynamic limit, before spending any effort on finding a pulse
sequence that would prepare the corresponding state on the device.

Once a state and an observable are deemed worth investigating, the same
`Operator` and `State` objects can be passed to an
[Expectation](../apidoc/_autosummary/pulser.backend.Expectation.rst)
observable and run through an emulator or QPU backend, as described in
[Execution on an Emulator](./backends.nblink) and
[Execution on a QPU](./qpu.nblink). See also
[State Preparation with the SLM Mask](./slm_mask.nblink) for one way to
prepare non-trivial initial states in practice.